In [0]:
# ==============================================
# TSI - FASE 2: CONSOLIDACIÓN INTERNACIONAL
# PASO 1: INSPECCIÓN DE ESQUEMAS DELTA
# ==============================================

TABLAS_FUENTE = {
    "istanbul": "workspace.tsi.istanbul_clean",
    "sao_paulo": "workspace.tsi.sao_paulo_clean",
    "madrid": "workspace.tsi.madrid_clean",
    "metr_la": "workspace.tsi.metr_la_clean",
    "pems_bay": "workspace.tsi.pems_bay_clean",
}

esquemas = {}

for fuente, tabla in TABLAS_FUENTE.items():

    df = spark.table(tabla)

    esquemas[fuente] = {
        campo.name: campo.dataType.simpleString()
        for campo in df.schema.fields
    }

    print(f"\n{'=' * 60}")
    print(f"FUENTE: {fuente.upper()}")
    print(f"TABLA: {tabla}")
    print(f"{'=' * 60}")

    for columna, tipo in esquemas[fuente].items():
        print(f"{columna:<30} {tipo}")

# ==============================================
# COMPARACIÓN GENERAL DE COLUMNAS
# ==============================================

todas_columnas = sorted(
    set().union(
        *(set(columnas) for columnas in esquemas.values())
    )
)

print("\n=== MATRIZ DE COMPATIBILIDAD ===")

print(
    f"{'COLUMNA':<30}"
    + "".join(f"{fuente:<15}" for fuente in TABLAS_FUENTE)
)

for columna in todas_columnas:

    print(
        f"{columna:<30}"
        + "".join(
            f"{esquemas[fuente].get(columna, 'NO EXISTE'):<15}"
            for fuente in TABLAS_FUENTE
        )
    )
    

In [0]:
# ==============================================
# TSI - VALIDACIÓN TEMPORAL DE LAS FUENTES
# ==============================================

from pyspark.sql import functions as F

print("=== CONFIGURACIÓN TEMPORAL DE SPARK ===")

zona_horaria = spark.conf.get("spark.sql.session.timeZone")

print(f"Zona horaria de sesión: {zona_horaria}")

print("\n=== TIPOS TEMPORALES ===")

for fuente, tabla in TABLAS_FUENTE.items():

    df = spark.table(tabla)

    tipo_timestamp = df.schema["timestamp"].dataType.simpleString()

    print(f"{fuente}: {tipo_timestamp}")

print("\n=== MUESTRAS TEMPORALES ===")

for fuente, tabla in TABLAS_FUENTE.items():

    print(f"\nFUENTE: {fuente.upper()}")

    spark.table(tabla).select(
        "timestamp",
        "ciudad",
        "fuente"
    ).limit(3).show(truncate=False)

print("\n=== PRUEBA DE CONVERSIÓN ESTAMBUL ===")

df_istanbul = spark.table("workspace.tsi.istanbul_clean")

df_istanbul.select(
    F.col("timestamp").alias("timestamp_original"),
    F.col("timestamp").cast("timestamp_ntz").alias("timestamp_convertido")
).limit(5).show(truncate=False)

In [0]:
# ==================================================
# TSI - FASE 2: CONSOLIDACIÓN INTERNACIONAL
# PASO 38: HOMOLOGACIÓN DE ESQUEMAS
# ==================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    DoubleType,
    LongType,
    TimestampNTZType
)

# Esquema canónico: unión de todas las columnas
# originales, conservando las métricas específicas.

columnas_string = [
    "ciudad",
    "fuente",
    "segmento_id",
    "tipo_via",
    "calidad_registro",
    "direction_fuente",
    "region_fuente",
    "passage_fuente",
]

columnas_double = [
    "latitud",
    "longitud",
    "velocidad_kmh",
    "flujo_veh_h",
    "ocupacion_pct",
    "densidad_veh_km",
    "tiempo_viaje_seg",
    "indice_congestion",
    "average_traffic_index",
    "congestion_longitud_m",
    "intensidad_fuente",
    "carga_fuente",
    "vmed_fuente",
    "error_fuente",
    "periodo_integracion_min",
    "valor_fuente",
]

columnas_bigint = [
    "minimum_traffic_index",
    "maximum_traffic_index",
]

tipos_canonicos = {
    "timestamp": "timestamp_ntz",
    "es_real": "boolean",
}

tipos_canonicos.update({
    columna: "string"
    for columna in columnas_string
})

tipos_canonicos.update({
    columna: "double"
    for columna in columnas_double
})

tipos_canonicos.update({
    columna: "bigint"
    for columna in columnas_bigint
})

# Verificar que ninguna columna de origen quede fuera
columnas_origen = set().union(
    *(set(columnas) for columnas in esquemas.values())
)

assert columnas_origen == set(tipos_canonicos), (
    "ERROR: Hay columnas sin contemplar en el esquema."
)

columnas_finales = list(tipos_canonicos.keys())

# Homologación sin modificar las tablas originales
dataframes_homologados = {}

for nombre_fuente, tabla in TABLAS_FUENTE.items():

    df_origen = spark.table(tabla)

    expresiones = []

    for columna in columnas_finales:

        tipo = tipos_canonicos[columna]

        if columna in df_origen.columns:
            expresion = F.col(columna).cast(tipo)

        else:
            expresion = F.lit(None).cast(tipo)

        expresiones.append(
            expresion.alias(columna)
        )

    df_homologado = df_origen.select(*expresiones)

    dataframes_homologados[nombre_fuente] = df_homologado

    print(
        f"{nombre_fuente}: "
        f"{len(df_homologado.columns)} columnas homologadas"
    )

print("\n=== ESQUEMA CANÓNICO ===")

dataframes_homologados["istanbul"].printSchema()

print("\nHOMOLOGACIÓN DEFINIDA CORRECTAMENTE")

In [0]:
# ==================================================
# TSI - FASE 2: CONSOLIDACIÓN INTERNACIONAL
# PASO 39: VALIDACIÓN DE CONVERSIONES
# ==================================================

from pyspark.sql import functions as F

errores_detectados = []

for nombre_fuente, tabla in TABLAS_FUENTE.items():

    print(f"\n=== VALIDANDO {nombre_fuente.upper()} ===")

    df_original = spark.table(tabla)

    verificaciones = []

    # Comprobar todas las columnas originales
    # que fueron convertidas a otro tipo.
    for columna in df_original.columns:

        tipo_original = (
            df_original.schema[columna]
            .dataType.simpleString()
        )

        tipo_destino = tipos_canonicos[columna]

        if tipo_original != tipo_destino:

            conversion = F.col(columna).cast(tipo_destino)

            # Detecta conversiones que pierden valores
            condicion_error = (
                F.col(columna).isNotNull()
                & conversion.isNull()
            )

            # Validación especial de timestamp:
            # compara representación textual en UTC.
            if columna == "timestamp":

                condicion_error = (
                    condicion_error
                    | (
                        F.col(columna).isNotNull()
                        & (
                            F.date_format(
                                F.col(columna),
                                "yyyy-MM-dd HH:mm:ss.SSSSSS"
                            )
                            !=
                            F.date_format(
                                conversion,
                                "yyyy-MM-dd HH:mm:ss.SSSSSS"
                            )
                        )
                    )
                )

            verificaciones.append(
                F.sum(
                    F.when(condicion_error, 1).otherwise(0)
                ).alias(columna)
            )

    if verificaciones:

        resultado = (
            df_original
            .agg(*verificaciones)
            .first()
            .asDict()
        )

        for columna, errores in resultado.items():

            print(
                f"{columna}: "
                f"{errores:,} conversiones problemáticas"
            )

            if errores > 0:
                errores_detectados.append(
                    (nombre_fuente, columna, errores)
                )

    else:
        print("No requiere conversiones de tipos.")

print("\n=== RESULTADO GENERAL ===")

if errores_detectados:

    print("SE DETECTARON CONVERSIONES PROBLEMÁTICAS")

    for fuente, columna, errores in errores_detectados:
        print(f"{fuente} | {columna} | {errores:,}")

else:
    print("VALIDACIÓN CORRECTA")
    print("No se detectaron pérdidas de valores por conversión.")

In [0]:
# ==================================================
# TSI - FASE 2: CONSOLIDACIÓN INTERNACIONAL
# PASO 40: UNIÓN DE LAS CINCO FUENTES
# ==================================================

from functools import reduce

TOTAL_ESPERADO = 41_770_880

# Recuperar los cinco DataFrames homologados
lista_dataframes = [
    dataframes_homologados[nombre]
    for nombre in TABLAS_FUENTE
]

# Unión vertical, conservando todos los registros
df_internacional = reduce(
    lambda df1, df2: df1.unionByName(df2),
    lista_dataframes
)

print("=== CONSOLIDACIÓN INTERNACIONAL ===")

print(f"Columnas: {len(df_internacional.columns)}")

assert len(df_internacional.columns) == 28, (
    "ERROR: El número de columnas no coincide."
)

# Comprobar cantidad total
total_consolidado = df_internacional.count()

print(f"Registros consolidados: {total_consolidado:,}")
print(f"Registros esperados:    {TOTAL_ESPERADO:,}")

assert total_consolidado == TOTAL_ESPERADO, (
    "ERROR: La consolidación no conserva todos los registros."
)

print("\n=== DISTRIBUCIÓN POR FUENTE ===")

df_internacional.groupBy(
    "ciudad",
    "fuente"
).count().orderBy(
    "ciudad",
    "fuente"
).show(truncate=False)

print("\nCONSOLIDACIÓN EN DATAFRAME CORRECTA")

In [0]:
# ==================================================
# TSI - FASE 2: CONSOLIDACIÓN INTERNACIONAL
# PASO 41: CREACIÓN DE TABLA DELTA CONSOLIDADA
# ==================================================

TABLA_CONSOLIDADA = (
    "workspace.tsi.international_observations"
)

TOTAL_ESPERADO = 41_770_880
COLUMNAS_ESPERADAS = 28

print("=== CREACIÓN DE TABLA DELTA ===")

# Verificar el esquema antes de escribir
assert len(df_internacional.columns) == COLUMNAS_ESPERADAS, (
    "ERROR: Número incorrecto de columnas."
)

# Verificar que no exista la tabla destino
if spark.catalog.tableExists(TABLA_CONSOLIDADA):

    print("La tabla consolidada ya existe.")
    print("No se realizará ninguna escritura.")

else:

    print("Creando tabla Delta consolidada...")

    (
        df_internacional.write
        .format("delta")
        .mode("append")
        .saveAsTable(TABLA_CONSOLIDADA)
    )

    print("Tabla Delta creada correctamente.")

print(f"\nTabla: {TABLA_CONSOLIDADA}")
print("Las cinco tablas originales permanecen intactas.")

In [0]:
# ==================================================
# TSI - FASE 2: CONSOLIDACIÓN INTERNACIONAL
# PASO 42: VALIDACIÓN FINAL DE TABLA DELTA
# ==================================================

from pyspark.sql import functions as F

TABLA_CONSOLIDADA = "workspace.tsi.international_observations"

TOTAL_ESPERADO = 41_770_880
COLUMNAS_ESPERADAS = 28

REGISTROS_ESPERADOS = {
    "istanbul_traffic_index": 3332,
    "sao_paulo_cet": 5223411,
    "madrid_traffic_points": 12512133,
    "metr_la": 7094304,
    "pems_bay": 16937700,
}

df_delta = spark.table(TABLA_CONSOLIDADA)

print("=== VALIDACIÓN GENERAL ===")

total = df_delta.count()
columnas = len(df_delta.columns)

print(f"Registros: {total:,}")
print(f"Columnas:  {columnas}")

assert total == TOTAL_ESPERADO, (
    "ERROR: El total de registros no coincide."
)

assert columnas == COLUMNAS_ESPERADAS, (
    "ERROR: El número de columnas no coincide."
)

assert df_delta.schema == df_internacional.schema, (
    "ERROR: El esquema consolidado no coincide."
)

print("\n=== VALIDACIÓN DEL FORMATO ===")

detalle = spark.sql(
    f"DESCRIBE DETAIL {TABLA_CONSOLIDADA}"
)

detalle.select(
    "format",
    "numFiles"
).show(truncate=False)

formato = detalle.select("format").first()[0]

assert formato == "delta", (
    "ERROR: La tabla no utiliza formato Delta."
)

print("\n=== DISTRIBUCIÓN POR FUENTE ===")

distribucion = (
    df_delta
    .groupBy("ciudad", "fuente")
    .count()
    .orderBy("ciudad", "fuente")
)

distribucion.show(truncate=False)

# Verificar las cantidades por fuente
resultados = (
    df_delta
    .groupBy("fuente")
    .count()
    .collect()
)

conteos_reales = {
    fila["fuente"]: fila["count"]
    for fila in resultados
}

assert conteos_reales == REGISTROS_ESPERADOS, (
    "ERROR: La distribución por fuente no coincide."
)

print("\n=== RESULTADO FINAL ===")
print("VALIDACIÓN CORRECTA")
print("La tabla consolidada conserva los registros esperados.")
print("Esquema y formato Delta verificados.")